### 1. Load Raw Dataset

In [ ]:
# 1) Load raw EN–NL dataset and normalize schema to a single `translation` column
import os
import random
import numpy as np
import torch
from datasets import DatasetDict, load_dataset
from transformers import set_seed

# Perf & cache (persist across restarts)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HOME", "../.cache/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "../.cache/hf/datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "../.cache/transformers")

# Faster matmul on Ada/L40S; consistent TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# Determinism (practical)
os.environ.setdefault("PYTHONHASHSEED", "42")
random.seed(42)
np.random.seed(42)
set_seed(42)
try:
    torch.use_deterministic_algorithms(False)  # keep False for speed; set True only if you need strict reproducibility
except Exception:
    pass

# Load: prefer curated; fall back to generic
try:
    dataset = load_dataset("yhavinga/open_subtitles_en_nl")
    source = "curated (yhavinga/open_subtitles_en_nl)"
except Exception:
    dataset = load_dataset("Helsinki-NLP/open_subtitles", lang1="en", lang2="nl")
    source = "generic (Helsinki-NLP/open_subtitles)"
print(f"Loaded: {source}")
assert isinstance(dataset, DatasetDict) and "train" in dataset, "Dataset must have a 'train' split"

# Normalize schema to: {'translation': {'en': str, 'nl': str}}
def _normalize_to_translation(batch):
    """Return a dict with a single 'translation' field."""
    if "translation" in batch:
        out = []
        for x in batch["translation"]:
            en = (x.get("en") or "").strip()
            nl = (x.get("nl") or "").strip()
            out.append({"en": en, "nl": nl})
        return {"translation": out}
    else:
        ens = batch.get("en", [])
        nls = batch.get("nl", [])
        out = []
        for en, nl in zip(ens, nls):
            out.append({"en": (en or "").strip(), "nl": (nl or "").strip()})
        return {"translation": out}

dataset = dataset.map(
    _normalize_to_translation,
    batched=True,
    remove_columns=[c for c in dataset["train"].column_names if c != "translation"],
    desc="Normalizing schema",
)

# quick size check
print("Train size:", len(dataset["train"]))

# safe peek without huge prints
ex0 = dataset["train"][0]["translation"]
print("\nRaw first example (EN→NL):")
print({"en": ex0["en"][:120], "nl": ex0["nl"][:120]})

print("\nFirst 5 pairs (truncated to 120 chars):")
for i in range(5):
    ex = dataset["train"][i]["translation"]
    print(f"{i+1:>2}. EN: {ex['en'][:120]}\n    NL: {ex['nl'][:120]}")

### 2. Basic Cleaning & Filtering

In [ ]:
# 2) Basic cleaning + langid + embedding filter (fast, parallel, stricter)
import hashlib
import os
import re
import unicodedata
import logging

import langid
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

# speed knobs
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
torch.backends.cudnn.benchmark = True  # let cuDNN pick fast kernels for ST encoder

NUM_PROC = max(1, (os.cpu_count() or 4) // 2)
CTRL = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069]")

def _strip_ctrl(s: str) -> str:
    s = CTRL.sub("", s or "")
    return unicodedata.normalize("NFKC", s).strip()

def _alnum_ratio(s: str) -> float:
    s = s or ""
    a = sum(ch.isalnum() for ch in s)
    return a / max(1, len(s))

# thresholds (stricter)
MIN_TOKENS = 3
LEN_RATIO_LOW = 1 / 3
LEN_RATIO_HIGH = 3.0
MAX_WORD_TOKENS = 180
MIN_ALNUM_RATIO = 0.60
EMB_THRESH = 0.60  # ↑ was 0.55; raise for cleaner pairs

# 1) normalize
dataset = dataset.map(
    lambda b: {
        "translation": [
            {"en": _strip_ctrl(x["en"]), "nl": _strip_ctrl(x["nl"])}
            for x in b["translation"]
        ]
    },
    batched=True,
    num_proc=NUM_PROC,
    desc="Normalize text",
)

# 2) lengths + alnum
def _feat(b):
    ens = [x["en"] for x in b["translation"]]
    nls = [x["nl"] for x in b["translation"]]
    return {
        "en_len": [len(t.split()) for t in ens],
        "nl_len": [len(t.split()) for t in nls],
        "en_alnum": [_alnum_ratio(t) for t in ens],
        "nl_alnum": [_alnum_ratio(t) for t in nls],
    }

dataset = dataset.map(_feat, batched=True, num_proc=NUM_PROC, desc="Lengths & alnum")

# 3) rule-based filter
def _keep(ex):
    el, nl = ex["en_len"], ex["nl_len"]
    if el < MIN_TOKENS or nl < MIN_TOKENS:
        return False
    if el > MAX_WORD_TOKENS or nl > MAX_WORD_TOKENS:
        return False
    r = el / max(1, nl)
    if not (LEN_RATIO_LOW < r < LEN_RATIO_HIGH):
        return False
    if ex["en_alnum"] < MIN_ALNUM_RATIO or ex["nl_alnum"] < MIN_ALNUM_RATIO:
        return False
    if not any(ch.isalpha() for ch in ex["translation"]["en"]):
        return False
    if not any(ch.isalpha() for ch in ex["translation"]["nl"]):
        return False
    return True

clean = dataset["train"].filter(_keep, num_proc=NUM_PROC, desc="Rule filter")

# 4) langid filter (batched, restricted, quiet)
langid.set_languages(["en", "nl"])  # restrict search space to EN/NL only
logging.getLogger("langid.langid").setLevel(logging.ERROR)

def _lang_mask(batch):
    ens = [x["en"] for x in batch["translation"]]
    nls = [x["nl"] for x in batch["translation"]]
    en_ok = [langid.classify(t)[0] == "en" for t in ens]
    nl_ok = [langid.classify(t)[0] == "nl" for t in nls]
    return {"keep_lang": [e and n for e, n in zip(en_ok, nl_ok)]}

# use moderate parallelism; per-example classification is Python-heavy
NUM_PROC_LANGID = min(8, max(1, (os.cpu_count() or 4) // 2))

tmp = clean.map(
    _lang_mask,
    batched=True,
    batch_size=4096,  # big batches to amortize overhead
    num_proc=NUM_PROC_LANGID,
    desc="LangID en/nl (batched)",
)
clean = tmp.filter(lambda k: k, input_columns=["keep_lang"]).remove_columns(["keep_lang"])

# 5) embedding similarity filter (GPU if available; larger encode batches)
emb_device = "cuda" if torch.cuda.is_available() else "cpu"
emb_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=emb_device
)
emb_model.max_seq_length = 256  # practical cap
EMB_BS = 2048 if emb_device == "cuda" else 256  # ↑ big batches on GPU

def _emb_sim_batch(batch):
    ens = [x["en"] for x in batch["translation"]]
    nls = [x["nl"] for x in batch["translation"]]
    e_en = emb_model.encode(
        ens, convert_to_tensor=True, normalize_embeddings=True,
        batch_size=EMB_BS, show_progress_bar=False
    )
    e_nl = emb_model.encode(
        nls, convert_to_tensor=True, normalize_embeddings=True,
        batch_size=EMB_BS, show_progress_bar=False
    )
    sims = util.cos_sim(e_en, e_nl).diagonal().float().cpu().numpy().tolist()
    return {"emb_sim": sims}

clean = clean.map(_emb_sim_batch, batched=True, batch_size=4096, desc="Embedding sims")
clean = clean.filter(lambda ex: ex["emb_sim"] >= EMB_THRESH, desc=f"Keep emb_sim >= {EMB_THRESH}")

# (Optional) keep top-quantile by similarity instead of hard threshold
q = 0.30  # drop bottom 30%
cut = float(np.quantile(clean["emb_sim"], q))
clean = clean.filter(lambda ex: ex["emb_sim"] >= cut, desc=f"Keep top {100*(1-q):.0f}% by emb_sim")

# 6) De-duplicate exact/near-exact pairs — vectorized (faster than a Python loop)
def _fp(batch):
    fps = []
    for x in batch["translation"]:
        en = (x["en"] or "").lower().strip()
        nl = (x["nl"] or "").lower().strip()
        fps.append(hashlib.blake2b((en + " ||| " + nl).encode("utf-8"), digest_size=12).hexdigest())
    return {"_fp": fps}

clean = clean.map(_fp, batched=True, desc="Fingerprint")

import pandas as pd
import datasets as hfds
df = clean.to_pandas()  # zero-copy via Arrow
df = df.drop_duplicates(subset=["_fp"], keep="first").reset_index(drop=True)
clean = hfds.Dataset.from_pandas(df, preserve_index=False).remove_columns(["_fp"])

# 7) shuffle + split once
clean = clean.shuffle(seed=42)
splits = clean.train_test_split(test_size=0.10, seed=42)
train_data, val_data = splits["train"], splits["test"]

# 8) drop helper cols to save RAM
drop_cols = [c for c in train_data.column_names if c not in ("translation",)]
train_data = train_data.remove_columns(drop_cols)
val_data = val_data.remove_columns([c for c in val_data.column_names if c not in ("translation",)])

print("Post-filter sizes:", len(train_data), len(val_data))

# 9) use all data (remove caps for best quality)
max_train = None
max_val = None
if max_train is not None and len(train_data) > max_train:
    train_data = train_data.select(range(max_train))
if max_val is not None and len(val_data) > max_val:
    val_data = val_data.select(range(max_val))
print("Final sizes:", len(train_data), len(val_data))

# Persist cleaned dataset to avoid recomputation on restarts
os.makedirs("../data", exist_ok=True)
clean.save_to_disk("../data/en_nl_clean.arrow")

### 3. Tokenizer Training (EN + NL together)

In [ ]:
# 3) Tokenizer training & load (Byte-level BPE, add_prefix_space=True)
import json
import os
import random

from tokenizers import ByteLevelBPETokenizer
from transformers import RobertaTokenizerFast

random.seed(42)

tok_dir = "../models/en_nl/tokenizer_v3"  # new folder to avoid mixing artifacts
os.makedirs(tok_dir, exist_ok=True)

# Build corpus (~2M lines == 1M pairs)
limit_pairs = min(1_000_000, len(train_data))
corpus_path = "all_sentences_en_nl.txt"

with open(corpus_path, "w", encoding="utf-8", newline="\n") as f:
    for i in range(limit_pairs):
        ex = train_data[i]["translation"]
        f.write((ex["en"]).strip() + "\n")
        f.write((ex["nl"]).strip() + "\n")

# Train byte-level BPE with RoBERTa/BART-friendly prefix spacing
bpe = ByteLevelBPETokenizer(add_prefix_space=True)
bpe.train(
    files=[corpus_path],
    vocab_size=32000,  # larger vocab for EN+NL coverage
    min_frequency=2,
    special_tokens=["<s>", "</s>", "<pad>", "<unk>"],
)

bpe.save_model(tok_dir)

with open(os.path.join(tok_dir, "tokenizer_config.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "model_type": "roberta",
            "unk_token": "<unk>",
            "bos_token": "<s>",
            "eos_token": "</s>",
            "pad_token": "<pad>",
            "add_prefix_space": True,
        },
        f,
        indent=2,
    )

tokenizer = RobertaTokenizerFast(
    vocab_file=os.path.join(tok_dir, "vocab.json"),
    merges_file=os.path.join(tok_dir, "merges.txt"),
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    add_prefix_space=True,
)
tokenizer.save_pretrained(tok_dir)

print("Vocab size:", tokenizer.vocab_size, "| add_prefix_space:", getattr(tokenizer, "add_prefix_space", None))

try:
    os.remove(corpus_path)
except OSError:
    pass

### 4. Model Definition

In [ ]:
# 4) Model definition (fresh init for the new tokenizer)
import torch
from transformers import BartConfig, BartForConditionalGeneration

# perf
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = "cuda" if torch.cuda.is_available() else "cpu"

config = BartConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=768,
    encoder_layers=8,
    decoder_layers=8,
    encoder_attention_heads=12,
    decoder_attention_heads=12,
    encoder_ffn_dim=3072,
    decoder_ffn_dim=3072,
    max_position_embeddings=512,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    dropout=0.1,
    attention_dropout=0.0,
    activation_dropout=0.0,
    activation_function="gelu",
    scale_embedding=True,
)

model = BartForConditionalGeneration(config)
model.resize_token_embeddings(tokenizer.vocab_size)

# trainer-friendly decoding defaults
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.forced_bos_token_id = None
model.config.forced_eos_token_id = tokenizer.eos_token_id
model.config.use_cache = False  # disable for gradient checkpointing / Trainer

# memory-efficiency
model.gradient_checkpointing_enable()

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {total_params/1e6:.1f}M (trainable {trainable_params/1e6:.1f}M)")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")

### 5. Tokenize train/val data for EN → NL

In [ ]:
# 5) Tokenize train/val for EN → NL (batched, fast, warning-free)
import os
max_len = 256

def tok_en2nl(batch):
    ens = [x["en"] for x in batch["translation"]]
    nls = [x["nl"] for x in batch["translation"]]
    enc_in = tokenizer(ens, truncation=True, max_length=max_len, padding=False)
    enc_out = tokenizer(text_target=nls, truncation=True, max_length=max_len, padding=False)
    return {
        "input_ids": enc_in["input_ids"],
        "attention_mask": enc_in["attention_mask"],
        "labels": enc_out["input_ids"],
    }

NUM_PROC_TOK = max(1, (os.cpu_count() or 8) // 2)
BATCH_SIZE_MAP = 1000  # larger batches reduce map() overhead

tokenized_train = train_data.map(
    tok_en2nl,
    batched=True,
    batch_size=BATCH_SIZE_MAP,
    num_proc=NUM_PROC_TOK,
    remove_columns=["translation"],
    desc="Tokenizing train",
)
tokenized_val = val_data.map(
    tok_en2nl,
    batched=True,
    batch_size=BATCH_SIZE_MAP,
    num_proc=NUM_PROC_TOK,
    remove_columns=["translation"],
    desc="Tokenizing val",
)

print("Tokenized sizes:", len(tokenized_train), len(tokenized_val))
print(
    "First example lens:",
    len(tokenized_train[0]["input_ids"]),
    len(tokenized_train[0]["attention_mask"]),
)

### 6. Training

In [ ]:
# 6) Training — faster per-epoch eval (subset + lighter beams), BLEU-tracked best checkpoint
from transformers import (
    DataCollatorForSeq2Seq,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from sacrebleu.metrics import BLEU, CHRF
from inspect import signature
import numpy as np
import torch

# = Speed up epoch-end evaluation =
#  - Evaluate on a capped subset (e.g., 10k examples) during training
#  - Use lighter decoding for eval: beams=3 and shorter max_new_tokens
#  - We'll still run full, strong decoding in Step 8 (final eval)

EVAL_CAP = 10_000  # cap eval set during training (adjustable)
eval_small = tokenized_val.select(range(min(EVAL_CAP, len(tokenized_val))))

# 6.a) Set generation defaults on the model (Trainer uses these during eval)
# Use lighter settings for *training-time* evaluation to save hours.
model.generation_config = GenerationConfig(
    max_new_tokens=96,                 # ↓ shorter decode for epoch-end eval
    num_beams=3,                       # ↓ cheaper beams for eval
    no_repeat_ngram_size=3,
    length_penalty=1.1,
    early_stopping=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# 6.b) Collator: dynamic pad + 8-multiple for tensor cores (speed)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    pad_to_multiple_of=8,
)

# 6.c) Robust metrics (handles tuples/tensors/logits/object arrays; clamps ids to vocab)
bleu = BLEU(effective_order=True, smooth_method="exp", lowercase=True)
chrf = CHRF(word_order=2)

def _unwrap_preds(x):
    if isinstance(x, (tuple, list)) and len(x) == 1:
        return x[0]
    return x

def _to_ids_2d(x):
    x = _unwrap_preds(x)
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    if not isinstance(x, np.ndarray):
        x = np.array(x, dtype=object)
    if x.ndim == 3:
        x = np.asarray(x).astype(np.float32, copy=False).argmax(-1)
    if not np.issubdtype(x.dtype, np.integer):
        try:
            x = x.astype(np.int64, copy=False)
        except Exception:
            x = np.vectorize(lambda z: int(z))(x)
    if x.ndim == 1:
        x = x[None, :]
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0
    vocab_hi = max(1, tokenizer.vocab_size) - 1
    x = np.where(x < 0, pad_id, x)
    x = np.where(x > vocab_hi, vocab_hi, x)
    return x.astype(np.int64, copy=False)

def compute_metrics(eval_pred):
    preds = getattr(eval_pred, "predictions", None)
    labels = getattr(eval_pred, "label_ids", None)
    if preds is None and isinstance(eval_pred, (tuple, list)) and len(eval_pred) >= 1:
        preds = eval_pred[0]
    if labels is None and isinstance(eval_pred, (tuple, list)) and len(eval_pred) >= 2:
        labels = eval_pred[1]

    pred_ids = _to_ids_2d(preds)
    label_ids = _to_ids_2d(labels)

    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0
    if label_ids.size > 0:
        label_ids = np.where(label_ids == -100, pad_id, label_ids)

    pred_texts = tokenizer.batch_decode(pred_ids.tolist(), skip_special_tokens=True)
    ref_texts  = tokenizer.batch_decode(label_ids.tolist(), skip_special_tokens=True)

    return {
        "bleu": bleu.corpus_score(pred_texts, [ref_texts]).score,
        "chrf2": chrf.corpus_score(pred_texts, [ref_texts]).score,
    }

# 6.d) Training arguments (cosine schedule, grad clip, bf16, best-by-BLEU)
sig = signature(Seq2SeqTrainingArguments.__init__)
strategy_kwargs = {}
if "evaluation_strategy" in sig.parameters:
    strategy_kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in sig.parameters:
    strategy_kwargs["eval_strategy"] = "epoch"

args = Seq2SeqTrainingArguments(
    output_dir="../models/en_nl_l40s_v3",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,      # effective ~128
    learning_rate=1.5e-4,
    weight_decay=0.01,
    num_train_epochs=10,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    label_smoothing_factor=0.15,
    # generation during eval (Trainer also uses model.generation_config above)
    predict_with_generate=True,
    generation_max_length=None,         # we rely on max_new_tokens in generation_config
    generation_num_beams=3,             # mirror the lighter eval beams
    # precision / perf
    bf16=True, fp16=False, tf32=True,
    dataloader_num_workers=12,          # small bump for throughput if CPU allows
    dataloader_pin_memory=True,
    group_by_length=True,
    optim="adamw_torch_fused",
    # logging / eval / save
    logging_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=True,        # restore BLEU-best (on eval subset)
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=3,
    save_safetensors=True,
    report_to=[],
    data_seed=42,
    **strategy_kwargs,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=eval_small,            # ← evaluate on the capped subset during training
    processing_class=tokenizer,         # future-proof
    data_collator=collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
eval_metrics = trainer.evaluate()
print("Eval (subset):", eval_metrics)

### 7. Inference helpers: Beam and optional MBR re-ranking

In [ ]:
# 7) Inference helpers: beam + optional MBR re-ranking
import re
import unicodedata
import torch
from sacrebleu.metrics import CHRF

CTRL = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069]")

def _clean(s: str) -> str:
    s = CTRL.sub("", s or "")
    return unicodedata.normalize("NFKC", s).strip()

@torch.inference_mode()
def translate_en2nl(
    texts,
    num_beams: int = 5,
    max_new_tokens: int = 128,
    min_new_tokens: int = 4,
    no_repeat_ngram_size: int = 3,
    length_penalty: float = 1.1,
    repetition_penalty: float = 1.05,
):
    """Standard high-quality beam decoding."""
    model.eval()
    src = [_clean(t) for t in texts]
    inputs = tokenizer(src, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
    gen_ids = model.generate(
        **inputs,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        no_repeat_ngram_size=no_repeat_ngram_size,
        length_penalty=length_penalty,
        repetition_penalty=repetition_penalty,
    )
    outs = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
    return [_clean(o) for o in outs]

# MBR decoding (reference-free): 1 strong beam + (K-1) samples; choose by mean chrF to others
@torch.inference_mode()
def translate_en2nl_mbr(
    texts,
    K: int = 10,               # total candidates per input (1 beam + K-1 samples)
    max_new_tokens: int = 128,
    top_p: float = 0.9,
    temperature: float = 0.7,
    batch_size: int = 32,
    seed: int | None = None,
    return_candidates: bool = False,    # if True, also return the n-best list per input
):
    """
    Reference-free MBR with chrF utility.
    - Mixes 1 strong beam with (K-1) diverse samples.
    - Picks the candidate with highest mean chrF vs. others.
    """
    model.eval()
    chrf = CHRF(word_order=2)
    cleaned = [_clean(t) for t in texts]
    results = []
    all_candidates = [] if return_candidates else None

    # Optional reproducible sampling
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    for start in range(0, len(cleaned), batch_size):
        batch = cleaned[start:start + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)

        # 1) One strong candidate via beam search
        beam_ids = model.generate(
            **enc,
            do_sample=False,
            num_beams=5,
            num_return_sequences=1,
            max_new_tokens=max_new_tokens,
            min_new_tokens=4,
            no_repeat_ngram_size=3,
            length_penalty=1.1,
        )
        # 2) (K-1) sampled candidates
        sK = max(1, K - 1)
        samp_ids = model.generate(
            **enc,
            do_sample=True,
            top_p=top_p,
            temperature=temperature,
            num_beams=1,
            num_return_sequences=sK,
            max_new_tokens=max_new_tokens,
            min_new_tokens=4,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
        )

        # Decode
        beam_txt = tokenizer.batch_decode(beam_ids, skip_special_tokens=True)
        samp_txt = tokenizer.batch_decode(samp_ids, skip_special_tokens=True)

        # Group per source: beam (1) + samples (sK)
        beam_groups = [beam_txt[i:i+1] for i in range(0, len(beam_txt), 1)]
        samp_groups = [samp_txt[i:i+sK] for i in range(0, len(samp_txt), sK)]

        # MBR select: candidate with highest mean chrF to others
        for beam1, samples in zip(beam_groups, samp_groups):
            cand_list = [_clean(t) for t in (beam1 + samples)]
            # de-duplicate while preserving order
            seen, uniq = set(), []
            for t in cand_list:
                if t and t not in seen:
                    seen.add(t); uniq.append(t)
            if not uniq:
                results.append("")
                if return_candidates:
                    all_candidates.append([])
                continue

            scores = []
            for i, c in enumerate(uniq):
                others = uniq[:i] + uniq[i+1:]
                mean_u = float(sum(chrf.sentence_score(c, [o]).score for o in others) / max(1, len(others)))
                scores.append(mean_u)

            best_idx = int(torch.tensor(scores).argmax().item())
            best = uniq[best_idx]
            results.append(best)
            if return_candidates:
                all_candidates.append(uniq)

    return (results, all_candidates) if return_candidates else results

### 8.a) BLEU evaluation

In [ ]:
# 8) FAST BLEU/chrF evaluation (greedy/partial, much quicker)
import torch
from sacrebleu.metrics import BLEU, CHRF
from transformers import GenerationConfig
import random

# --- Speed mode toggles ---
FAST = True                # set False for full-strength eval
SAMPLE_N = 2000            # how many validation examples to sample (only if FAST=True)

# --- Decoding knobs (FAST vs FULL) ---
if FAST:
    N = SAMPLE_N
    GEN_MAX_NEW = 96       # shorter decode
    NUM_BEAMS = 1          # greedy (fastest)
    BATCH_SIZE = 128       # bigger batch if VRAM allows
    NO_REPEAT = 0          # disable constraint for speed
else:
    N = None               # full validation set
    GEN_MAX_NEW = 128
    NUM_BEAMS = 5
    BATCH_SIZE = 64
    NO_REPEAT = 3

# Ensure strong-but-overridden generation config (we pass args explicitly anyway)
model.generation_config = GenerationConfig(
    max_new_tokens=GEN_MAX_NEW,
    num_beams=NUM_BEAMS,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# --- Build (optionally sampled) eval slice ---
if N is None or N >= len(val_data):
    idxs = list(range(len(val_data)))
else:
    random.seed(42)
    idxs = random.sample(range(len(val_data)), k=N)

src_texts = [val_data[i]["translation"]["en"] for i in idxs]
references = [val_data[i]["translation"]["nl"] for i in idxs]

device = next(model.parameters()).device
hypotheses = []
model.eval()

# Optional: enable flash/mem-efficient attention where possible
try:
    torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=True)
except Exception:
    pass

@torch.inference_mode()
def _generate_batch(texts):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
        gen_ids = model.generate(
            **enc,
            use_cache=True,                 # speed up decoding
            num_beams=NUM_BEAMS,
            do_sample=False,                # greedy when beams=1
            max_new_tokens=GEN_MAX_NEW,
            min_new_tokens=2,
            no_repeat_ngram_size=NO_REPEAT,
            length_penalty=1.0 if NUM_BEAMS == 1 else 1.1,
        )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

for i in range(0, len(src_texts), BATCH_SIZE):
    batch_src = src_texts[i : i + BATCH_SIZE]
    outs = _generate_batch(batch_src)
    hypotheses.extend([s.strip() for s in outs])

bleu = BLEU(effective_order=True, smooth_method="exp", lowercase=True)
chrf = CHRF(word_order=2)

print(
    ("[FAST] " if FAST else "[FULL] ")
    + f"BLEU = {bleu.corpus_score(hypotheses, [references]).score:.2f} "
    + f"| chrF2 = {chrf.corpus_score(hypotheses, [references]).score:.2f} "
    + f"| n={len(src_texts)} | beams={NUM_BEAMS} | max_new={GEN_MAX_NEW} | bs={BATCH_SIZE}"
)

for s, hyp, ref in list(zip(src_texts, hypotheses, references))[:5]:
    print("\nEN:", s)
    print("NL(hyp):", hyp)
    print("NL(ref):", ref)

### 8.b) re-score checkpoints & pick true best

In [ ]:
# Re-evaluate recent checkpoints with strong decoding, pick true best by BLEU, and save
import os, re, glob
from pathlib import Path
from transformers import GenerationConfig, Seq2SeqTrainer

ckpt_root = Path("../models/en_nl_l40s_v3")
save_dir  = Path("../models/en_nl_l40s_final_v3_true_best")
save_dir.mkdir(parents=True, exist_ok=True)

# Strong decoding config (full-strength)
model.generation_config = GenerationConfig(
    max_new_tokens=128,
    num_beams=5,
    no_repeat_ngram_size=3,
    length_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Collect checkpoint paths (limit to last N to save time)
def step_num(p): 
    m = re.search(r"checkpoint-(\d+)", p)
    return int(m.group(1)) if m else -1

checkpoints = sorted(glob.glob(str(ckpt_root / "checkpoint-*")), key=step_num)
# evaluate all or just the last few:
checkpoints = checkpoints[-3:] if len(checkpoints) > 3 else checkpoints
print("Will evaluate:", checkpoints)

best_bleu, best_ckpt = -1.0, None
for ckpt in checkpoints:
    tmp_model = type(model).from_pretrained(ckpt).to(next(model.parameters()).device)
    tmp_model.generation_config = model.generation_config
    tmp_trainer = Seq2SeqTrainer(
        model=tmp_model,
        args=trainer.args,                 # reuse args (batch size, device, etc.)
        eval_dataset=tokenized_val,        # FULL validation set
        processing_class=tokenizer,
        data_collator=trainer.data_collator,
        compute_metrics=trainer.compute_metrics,
    )
    metrics = tmp_trainer.evaluate()
    # key may be "eval_bleu" depending on HF; handle both
    bleu_score = metrics.get("eval_bleu", metrics.get("bleu", -1.0))
    print(f"Checkpoint {ckpt} BLEU: {bleu_score:.2f}")
    if bleu_score > best_bleu:
        best_bleu, best_ckpt = bleu_score, ckpt

print(f"\nBest checkpoint: {best_ckpt} (BLEU {best_bleu:.2f})")

# Load the best and save as your final model
best_model = type(model).from_pretrained(best_ckpt).to(next(model.parameters()).device)
best_model.generation_config = model.generation_config
best_model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

# (optional) save training args/state for provenance
trainer.save_state()
with open(save_dir / "training_args.json", "w") as f:
    f.write(trainer.args.to_json_string())

print("Saved true-best model to:", save_dir)

### 9. Save the fine-tuned EN→NL model + tokenizer

In [ ]:
# 9) Save model & tokenizer
import os
save_dir = "../models/en_nl_l40s_final_v3"
os.makedirs(save_dir, exist_ok=True)

# Save weights, config, and tokenizer
model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

# Also persist the generation config used during eval/inference
try:
    model.generation_config.to_json_file(os.path.join(save_dir, "generation_config.json"))
except Exception:
    pass

print("Saved to:", save_dir)

### 10. Translate

In [ ]:
# 10) Translate a file (EN -> NL)
from pathlib import Path
import pandas as pd
import torch

INPUT_PATH = Path("../data/STT_Assembly.xlsx")
OUTPUT_CSV = Path("translations_en2nl.csv")  # semicolon-delimited
BATCH_SIZE = 64

# 1) Load
if INPUT_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(INPUT_PATH, dtype=str)
else:
    df = pd.read_csv(INPUT_PATH, sep=None, engine="python", dtype=str)

# 2) Choose the source column
df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed") or c in {"S","I","D"}],
             errors="ignore")
sent_col = next((c for c in df.columns if str(c).strip().lower() == "sentence"), df.columns[0])
sentences = df[sent_col].astype(str).fillna("").tolist()

# 3) Translate in batches (beam). For MBR, swap the function below.
def translate_batch_beam(texts):
    model.eval()
    tok = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        ids = model.generate(
            **tok,
            num_beams=5,
            max_new_tokens=128,
            min_new_tokens=4,         # align with inference helpers
            no_repeat_ngram_size=3,
            length_penalty=1.1,
            repetition_penalty=1.05,
        )
    return tokenizer.batch_decode(ids, skip_special_tokens=True)

outputs = []
for i in range(0, len(sentences), BATCH_SIZE):
    batch = sentences[i : i + BATCH_SIZE]
    outputs.extend(translate_batch_beam(batch))
    # For extra quality (slower), use MBR instead:
    # outputs.extend(translate_en2nl_mbr(batch, K=10))

# 4) Save clean CSV (semicolon)
def clean(s: str) -> str:
    return str(s).replace("\r", " ").replace("\n", " ").replace(";", ",").strip()

out = pd.DataFrame({"Sentence": [clean(s) for s in sentences],
                    "NL": [clean(t) for t in outputs]})
out.to_csv(OUTPUT_CSV, sep=";", index=False, encoding="utf-8-sig")
print("Wrote:", OUTPUT_CSV)

### 11. Polishing

In [ ]:
# 11) Polishing (domain glossary + idioms + spacing)
import re
import unicodedata
import pandas as pd

IN_CSV = "translations_en2nl.csv"
OUT_CSV = "translations_en2nl_polished_final.csv"

df = pd.read_csv(IN_CSV, sep=";", dtype=str)
df["Sentence"] = df["Sentence"].astype(str)
df["NL"] = df["NL"].astype(str)

def _norm(text: str) -> str:
    # Unicode normalize + strip common control/invisible chars
    text = unicodedata.normalize("NFKC", text or "")
    text = re.sub(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069]", "", text)
    return text.strip()

def sub_ci(text, pattern, repl):
    return re.sub(pattern, repl, text, flags=re.IGNORECASE)

def tidy_spaces(text):
    # collapse spaces before punctuation and multiple spaces
    text = re.sub(r"\s+([,.:;!?…])", r"\1", text)
    text = re.sub(r"\s{2,}", " ", text)
    # normalize spaced ellipses/quotes/dashes a bit
    text = re.sub(r"\s*…\s*", "…", text)
    text = re.sub(r"\s*—\s*", " — ", text)
    return text.strip()

GLOSS = [
    (r"\btribal council\b", "stamraad"),
    (r"\btribe(s)?\b", "stam"),
    (r"\balliance\b", "alliantie"),
    (r"\bcastaway(s)?\b", "deelnemer"),
    (r"\breward challenge\b", "beloningsproef"),
    (r"\bimmunity challenge\b", "immuniteitsproef"),
    (r"\b(hidden )?immunity idol\b", "verborgen immuniteitsbeeldje"),
    (r"\bimmunity idol\b", "immuniteitsbeeldje"),
    (r"\bimmunity\b", "immuniteit"),
    (r"\bmerge\b", "samensmelting"),
    (r"\bvote (?:off|out)\b", "wegstemmen"),
    (r"\bblindside(d|s)?\b", "onverwacht wegstemmen"),
    (r"\bcamp\b", "kamp"),
    (r"\btorch\b", "fakkel"),
    (r"\bfinal tribal council\b", "finale stamraad"),
]

FIXES = [
    (r"Wat voorafging in Australisch Survivor", "Vorige keer bij Australian Survivor"),
    (r"\bop de outs\b", "aan de zijlijn"),
    (r"\bouts\b", "buitenspel"),
    (r"\bsinds een dag\b", "sinds dag één"),
    (r"\bblindsid(?:e|en|ed)\b", "overvallen"),
    (r"\bverblinden\b", "overvallen"),
    (r"\bdat hing in de balans\b", "dat op het spel stond"),
    (r"^Survivors klaar$", "Survivors, klaar?"),
    (r"^Ga maar naar huis$", "Start!"),
    (r"\bsmoker\b", "roker"),
    (r"\buitdaging\b", "proef"),
    (r"\bscreen[- ]?cap\b", "schermafbeelding"),
    (r"\bgrill\b", "Grill"),
    (r"\bz'n\b", "zijn"),
]

STYLE = [
    (r"\bOK\b", "Oké"),
    (r"\boke\b", "oké"),
    (r"\.\.\.", "…"),
]

def polish(text: str) -> str:
    if not text:
        return text
    text = _norm(text)
    for pat, rep in GLOSS:
        text = sub_ci(text, pat, rep)
    for pat, rep in FIXES:
        text = sub_ci(text, pat, rep)
    for pat, rep in STYLE:
        text = sub_ci(text, pat, rep)
    return tidy_spaces(text)

df["NL"] = df["NL"].map(polish)
df.to_csv(OUT_CSV, sep=";", index=False, encoding="utf-8-sig")
print("Wrote:", OUT_CSV)

# Part 2: NL → EN Model

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0") 
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

### 1. Load Raw Dataset:

In [ ]:
# 1) Load raw EN–NL dataset and normalize schema to a single `translation` column
import os
import random
import numpy as np
import torch
from datasets import DatasetDict, load_dataset
from transformers import set_seed

# Perf & cache (persist across restarts)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HOME", "../.cache/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "../.cache/hf/datasets")
os.environ.setdefault("TRANSFORMERS_CACHE", "../.cache/transformers")

# Faster matmul on Ada/L40S; consistent TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# Determinism (practical)
os.environ.setdefault("PYTHONHASHSEED", "42")
random.seed(42)
np.random.seed(42)
set_seed(42)
try:
    torch.use_deterministic_algorithms(False)  # keep False for speed; set True only if you need strict reproducibility
except Exception:
    pass

# Load: prefer curated; fall back to generic
try:
    dataset = load_dataset("yhavinga/open_subtitles_en_nl")
    source = "curated (yhavinga/open_subtitles_en_nl)"
except Exception:
    dataset = load_dataset("Helsinki-NLP/open_subtitles", lang1="en", lang2="nl")
    source = "generic (Helsinki-NLP/open_subtitles)"
print(f"Loaded: {source}")
assert isinstance(dataset, DatasetDict) and "train" in dataset, "Dataset must have a 'train' split"

# Normalize schema to: {'translation': {'en': str, 'nl': str}}
def _normalize_to_translation(batch):
    """Return a dict with a single 'translation' field."""
    if "translation" in batch:
        out = []
        for x in batch["translation"]:
            en = (x.get("en") or "").strip()
            nl = (x.get("nl") or "").strip()
            out.append({"en": en, "nl": nl})
        return {"translation": out}
    else:
        ens = batch.get("en", [])
        nls = batch.get("nl", [])
        out = []
        for en, nl in zip(ens, nls):
            out.append({"en": (en or "").strip(), "nl": (nl or "").strip()})
        return {"translation": out}

dataset = dataset.map(
    _normalize_to_translation,
    batched=True,
    remove_columns=[c for c in dataset["train"].column_names if c != "translation"],
    desc="Normalizing schema",
)

# quick size check
print("Train size:", len(dataset["train"]))

# safe peek without huge prints
ex0 = dataset["train"][0]["translation"]
print("\nRaw first example (NL→EN):")
print({"nl": ex0["nl"][:120], "en": ex0["en"][:120]})

print("\nFirst 5 pairs (truncated to 120 chars):")
for i in range(5):
    ex = dataset["train"][i]["translation"]
    print(f"{i+1:>2}. NL: {ex['nl'][:120]}\n    EN: {ex['en'][:120]}")

### 2. Basic Cleaning & Filtering

In [ ]:
# 2) Basic cleaning + langid + embedding filter
import hashlib, os, re, unicodedata, logging
import langid
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# Performance setup:
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
torch.backends.cudnn.benchmark = True          # kernel autotune
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

NUM_PROC = max(1, (os.cpu_count() or 4) - 2)
CTRL = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069]")

def _strip_ctrl(s: str) -> str:
    s = CTRL.sub("", s or "")
    return unicodedata.normalize("NFKC", s).strip()

def _alnum_ratio(s: str) -> float:
    s = s or ""
    a = sum(ch.isalnum() for ch in s)
    return a / max(1, len(s))

# Thresholds:
MIN_TOKENS = 3
LEN_RATIO_LOW, LEN_RATIO_HIGH = 1/3, 3.0
MAX_WORD_TOKENS = 180
MIN_ALNUM_RATIO = 0.60
EMB_THRESH = 0.60   # keep only a hard similarity floor; no quantile drop

# 1) Normalize
dataset = dataset.map(
    lambda b: {
        "translation": [
            {"en": _strip_ctrl(x["en"]), "nl": _strip_ctrl(x["nl"])}
            for x in b["translation"]
        ]
    },
    batched=True,
    num_proc=NUM_PROC,
    desc="Normalize text",
)

# 2) Compute length & alphanumeric ratio
def _feat(b):
    ens = [x["en"] for x in b["translation"]]
    nls = [x["nl"] for x in b["translation"]]
    return {
        "en_len": [len(t.split()) for t in ens],
        "nl_len": [len(t.split()) for t in nls],
        "en_alnum": [_alnum_ratio(t) for t in ens],
        "nl_alnum": [_alnum_ratio(t) for t in nls],
    }

dataset = dataset.map(_feat, batched=True, num_proc=NUM_PROC, desc="Lengths & alnum")

# 3) Rule-based filter
def _keep(ex):
    el, nl = ex["en_len"], ex["nl_len"]
    if el < MIN_TOKENS or nl < MIN_TOKENS: return False
    if el > MAX_WORD_TOKENS or nl > MAX_WORD_TOKENS: return False
    r = el / max(1, nl)
    if not (LEN_RATIO_LOW < r < LEN_RATIO_HIGH): return False
    if ex["en_alnum"] < MIN_ALNUM_RATIO or ex["nl_alnum"] < MIN_ALNUM_RATIO: return False
    if not any(ch.isalpha() for ch in ex["translation"]["en"]): return False
    if not any(ch.isalpha() for ch in ex["translation"]["nl"]): return False
    return True

clean = dataset["train"].filter(_keep, num_proc=NUM_PROC, desc="Rule filter")

# 4) LangID filter (restricted to EN/NL)
langid.set_languages(["en", "nl"])
logging.getLogger("langid.langid").setLevel(logging.ERROR)

def _lang_mask(batch):
    ens = [x["en"] for x in batch["translation"]]
    nls = [x["nl"] for x in batch["translation"]]
    en_ok = [langid.classify(t)[0] == "en" for t in ens]
    nl_ok = [langid.classify(t)[0] == "nl" for t in nls]
    return {"keep_lang": [e and n for e, n in zip(en_ok, nl_ok)]}

tmp = clean.map(
    _lang_mask,
    batched=True,
    batch_size=4096,
    num_proc=min(8, NUM_PROC),
    desc="LangID en/nl (batched)",
)
clean = tmp.filter(lambda k: k, input_columns=["keep_lang"]).remove_columns(["keep_lang"])

# 5) Embedding similarity filter (GPU-accelerated, conservative batches/slabs)
emb_device = "cuda" if torch.cuda.is_available() else "cpu"
emb_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=emb_device
)
emb_model.max_seq_length = 256

# Conservative to avoid OOM/fragmentation while keeping throughput
ENC_BS = 2048 if emb_device == "cuda" else 256
CHUNK  = 50_000

ens = [x["en"] for x in clean["translation"]]
nls = [x["nl"] for x in clean["translation"]]
sims = np.empty(len(clean), dtype=np.float32)

@torch.inference_mode()
def embed_texts(texts):
    return emb_model.encode(
        texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=ENC_BS,
        show_progress_bar=False,
    )

for start in tqdm(range(0, len(clean), CHUNK), desc="Embedding sims (chunked, GPU)"):
    end = min(start + CHUNK, len(clean))
    e_en = embed_texts(ens[start:end])
    e_nl = embed_texts(nls[start:end])
    sims[start:end] = (e_en * e_nl).sum(dim=1).float().cpu().numpy()
    del e_en, e_nl
    if emb_device == "cuda":
        torch.cuda.empty_cache()

clean = clean.add_column("emb_sim", sims.tolist())
clean = clean.filter(lambda ex: ex["emb_sim"] >= EMB_THRESH, desc=f"Keep emb_sim ≥ {EMB_THRESH}")

# 6) De-duplicate (exact/near-exact)
def _fp(batch):
    fps = []
    for x in batch["translation"]:
        en = (x["en"] or "").lower().strip()
        nl = (x["nl"] or "").lower().strip()
        fps.append(hashlib.blake2b((en + " ||| " + nl).encode("utf-8"), digest_size=12).hexdigest())
    return {"_fp": fps}

clean = clean.map(_fp, batched=True, desc="Fingerprint")

import pandas as pd, datasets as hfds
df = clean.to_pandas()
df = df.drop_duplicates(subset=["_fp"], keep="first").reset_index(drop=True)
clean = hfds.Dataset.from_pandas(df, preserve_index=False).remove_columns(["_fp"])

# 7) Shuffle + split
clean = clean.shuffle(seed=42)
splits = clean.train_test_split(test_size=0.10, seed=42)
train_data, val_data = splits["train"], splits["test"]

# 8) Drop helper cols
drop_cols = [c for c in train_data.column_names if c not in ("translation",)]
train_data = train_data.remove_columns(drop_cols)
val_data = val_data.remove_columns([c for c in val_data.column_names if c not in ("translation",)])

print("Post-filter sizes:", len(train_data), len(val_data))

# 9) Use ALL available data (no final caps for quality)
MAX_TRAIN = None
MAX_VAL   = None
if MAX_TRAIN and len(train_data) > MAX_TRAIN:
    train_data = train_data.select(range(MAX_TRAIN))
if MAX_VAL and len(val_data) > MAX_VAL:
    val_data = val_data.select(range(MAX_VAL))
print("Final sizes:", len(train_data), len(val_data))

# Persist cleaned dataset (quality version)
os.makedirs("../data", exist_ok=True)
clean.save_to_disk("../data/nl_en_clean.arrow")

### 3. Re-use same tokenizer

In [ ]:
# 3) Tokenizer training & load (Byte-level BPE, add_prefix_space=True, quality-first)
import json
import os
import random
from pathlib import Path

from tokenizers import ByteLevelBPETokenizer
from transformers import RobertaTokenizerFast

random.seed(42)

tok_dir = Path("../models/nl_en/tokenizer_v3")  # separate folder for NL→EN
tok_dir.mkdir(parents=True, exist_ok=True)

vocab_file = tok_dir / "vocab.json"
merges_file = tok_dir / "merges.txt"
tok_cfg_file = tok_dir / "tokenizer_config.json"

# If tokenizer already exists, just load it (avoid retraining)
if vocab_file.exists() and merges_file.exists():
    tokenizer = RobertaTokenizerFast(
        vocab_file=str(vocab_file),
        merges_file=str(merges_file),
        bos_token="<s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
        add_prefix_space=True,
    )
    tokenizer.save_pretrained(str(tok_dir))
    print(f"[Tokenizer] Loaded existing tokenizer from {tok_dir}")
else:
    # Build corpus (~2M lines == 1M pairs) from cleaned train split
    limit_pairs = min(1_000_000, len(train_data))
    corpus_path = Path("all_sentences_nl_en.txt")

    with corpus_path.open("w", encoding="utf-8", newline="\n") as f:
        for i in range(limit_pairs):
            ex = train_data[i]["translation"]
            # include BOTH languages so tokenizer covers both well
            f.write((ex["nl"] or "").strip() + "\n")
            f.write((ex["en"] or "").strip() + "\n")

    # Train byte-level BPE with RoBERTa/BART-friendly prefix spacing
    bpe = ByteLevelBPETokenizer(add_prefix_space=True)
    bpe.train(
        files=[str(corpus_path)],
        vocab_size=32000,                 # quality: ample coverage for EN+NL
        min_frequency=2,
        special_tokens=["<s>", "</s>", "<pad>", "<unk>"],
    )

    bpe.save_model(str(tok_dir))

    with tok_cfg_file.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "model_type": "roberta",
                "unk_token": "<unk>",
                "bos_token": "<s>",
                "eos_token": "</s>",
                "pad_token": "<pad>",
                "add_prefix_space": True,
            },
            f,
            indent=2,
        )

    tokenizer = RobertaTokenizerFast(
        vocab_file=str(vocab_file),
        merges_file=str(merges_file),
        bos_token="<s>",
        eos_token="</s>",
        unk_token="<unk>",
        pad_token="<pad>",
        add_prefix_space=True,
    )
    tokenizer.save_pretrained(str(tok_dir))

    print("Vocab size:", tokenizer.vocab_size, "| add_prefix_space:", getattr(tokenizer, "add_prefix_space", None))

    try:
        corpus_path.unlink(missing_ok=True)
    except Exception:
        pass

### 4. Model definition (same arch)

In [ ]:
# 4) Model definition (fresh init for the new tokenizer; quality-first, GPU-friendly)
import torch
from transformers import BartConfig, BartForConditionalGeneration

# Perf toggles (safe for Ada/Ampere+)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

device = "cuda" if torch.cuda.is_available() else "cpu"

config = BartConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=768,
    encoder_layers=8,
    decoder_layers=8,
    encoder_attention_heads=12,
    decoder_attention_heads=12,
    encoder_ffn_dim=3072,
    decoder_ffn_dim=3072,
    max_position_embeddings=512,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    dropout=0.1,
    attention_dropout=0.0,
    activation_dropout=0.0,
    activation_function="gelu",
    scale_embedding=True,        # better stability for larger vocab
    normalize_before=True,       # pre-LN (Bart supports this; improves stability)
)

model = BartForConditionalGeneration(config)

# Ensure embeddings match tokenizer size (no-op if already equal)
model.resize_token_embeddings(tokenizer.vocab_size)

# Trainer/generation-friendly defaults
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.forced_bos_token_id = None
model.config.forced_eos_token_id = tokenizer.eos_token_id
model.config.use_cache = False          # disable during training (needed with checkpointing)

# Memory-efficiency & longer context training
model.gradient_checkpointing_enable()   # reduce activation memory at small compute cost

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {total_params/1e6:.1f}M (trainable {trainable_params/1e6:.1f}M)")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")
print("Config: "
      f"d_model={config.d_model}, layers(enc/dec)={config.encoder_layers}/{config.decoder_layers}, "
      f"heads(enc/dec)={config.encoder_attention_heads}/{config.decoder_attention_heads}")

### 5. Tokenize train/val for NL → EN

In [ ]:
# 5) Tokenize train/val for NL → EN
import os

# Use the same context length as the EN→NL run for consistency
max_len = 256

def tok_nl2en(batch):
    nls = [x["nl"] for x in batch["translation"]]   # source (NL)
    ens = [x["en"] for x in batch["translation"]]   # target (EN)
    enc_in  = tokenizer(nls, truncation=True, max_length=max_len, padding=False)
    enc_out = tokenizer(text_target=ens, truncation=True, max_length=max_len, padding=False)
    return {
        "input_ids": enc_in["input_ids"],
        "attention_mask": enc_in["attention_mask"],
        "labels": enc_out["input_ids"],
    }

# Balanced parallelism: enough to be fast without starving I/O
NUM_PROC_TOK   = max(1, (os.cpu_count() or 8) // 2)
BATCH_SIZE_MAP = 1000  # practical batch size

# Remove only what's present to avoid arrow warnings
rm_cols_train = [c for c in train_data.column_names if c != "translation"]
rm_cols_val   = [c for c in val_data.column_names if c != "translation"]

tokenized_train = train_data.map(
    tok_nl2en,
    batched=True,
    batch_size=BATCH_SIZE_MAP,
    num_proc=NUM_PROC_TOK,
    remove_columns=rm_cols_train,
    desc="Tokenizing train (NL→EN)",
)

tokenized_val = val_data.map(
    tok_nl2en,
    batched=True,
    batch_size=BATCH_SIZE_MAP,
    num_proc=NUM_PROC_TOK,
    remove_columns=rm_cols_val,
    desc="Tokenizing val (NL→EN)",
)

print("Tokenized sizes:", len(tokenized_train), len(tokenized_val))
print("First example lens:",
      len(tokenized_train[0]["input_ids"]),
      len(tokenized_train[0]["attention_mask"]))

### 6. Training (same set-up; new output_dir)

In [ ]:
# 6) Training - quality-oriented, full eval, BLEU-tracked best checkpoint
from transformers import (
    DataCollatorForSeq2Seq,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from sacrebleu.metrics import BLEU, CHRF
from inspect import signature
import numpy as np
import torch

# Eval set:
EVAL_CAP = None  # set e.g. 10_000 to cap during training; None = full val set
eval_dataset = tokenized_val if EVAL_CAP is None else tokenized_val.select(range(min(EVAL_CAP, len(tokenized_val))))

# Generation defaults used by Trainer during eval (stronger decoding)
model.generation_config = GenerationConfig(
    max_new_tokens=128,
    num_beams=5,                   # beam search for quality
    no_repeat_ngram_size=3,
    length_penalty=1.1,
    early_stopping=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Collator: dynamic pad & 8-multiple (tensor cores)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",
    pad_to_multiple_of=8,
)

# Metrics (robust to shapes / dtypes)
bleu = BLEU(effective_order=True, smooth_method="exp", lowercase=True)
chrf = CHRF(word_order=2)

def _unwrap_preds(x):
    if isinstance(x, (tuple, list)) and len(x) == 1:
        return x[0]
    return x

def _to_ids_2d(x):
    x = _unwrap_preds(x)
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    if not isinstance(x, np.ndarray):
        x = np.array(x, dtype=object)
    if x.ndim == 3:  # logits
        x = np.asarray(x).astype(np.float32, copy=False).argmax(-1)
    if not np.issubdtype(x.dtype, np.integer):
        try:
            x = x.astype(np.int64, copy=False)
        except Exception:
            x = np.vectorize(lambda z: int(z))(x)
    if x.ndim == 1:
        x = x[None, :]
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0
    vocab_hi = max(1, tokenizer.vocab_size) - 1
    x = np.where(x < 0, pad_id, x)
    x = np.where(x > vocab_hi, vocab_hi, x)
    return x.astype(np.int64, copy=False)

def compute_metrics(eval_pred):
    preds = getattr(eval_pred, "predictions", None)
    labels = getattr(eval_pred, "label_ids", None)
    if preds is None and isinstance(eval_pred, (tuple, list)) and len(eval_pred) >= 1:
        preds = eval_pred[0]
    if labels is None and isinstance(eval_pred, (tuple, list)) and len(eval_pred) >= 2:
        labels = eval_pred[1]

    pred_ids  = _to_ids_2d(preds)
    label_ids = _to_ids_2d(labels)

    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0
    if label_ids.size > 0:
        label_ids = np.where(label_ids == -100, pad_id, label_ids)

    pred_texts = tokenizer.batch_decode(pred_ids.tolist(),  skip_special_tokens=True)
    ref_texts  = tokenizer.batch_decode(label_ids.tolist(), skip_special_tokens=True)

    return {
        "bleu":  bleu.corpus_score(pred_texts, [ref_texts]).score,
        "chrf2": chrf.corpus_score(pred_texts, [ref_texts]).score,
    }

# Version-compatible args (evaluation_strategy vs eval_strategy)
sig = signature(Seq2SeqTrainingArguments.__init__)
strategy_kwargs = {}
if "evaluation_strategy" in sig.parameters:
    strategy_kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in sig.parameters:
    strategy_kwargs["eval_strategy"] = "epoch"

# Precision flags chosen by capability (prefer bf16 if available)
use_cuda = torch.cuda.is_available()
cap_major = torch.cuda.get_device_capability()[0] if use_cuda else 0
use_bf16 = bool(use_cuda and cap_major >= 8)  # Ampere/Ada+
use_fp16 = bool(use_cuda and not use_bf16)    # fallback to fp16 if no bf16
use_tf32 = bool(use_cuda)

args = Seq2SeqTrainingArguments(
    output_dir="../models/nl_en_l40s_v3",
    per_device_train_batch_size=32,      
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,       # effective batch ≈ 128
    learning_rate=1.5e-4,
    weight_decay=0.01,
    num_train_epochs=10,                 # longer training for BLEU
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    label_smoothing_factor=0.15,
    # generation during eval
    predict_with_generate=True,
    generation_max_length=None,          # use max_new_tokens from generation_config
    generation_num_beams=5,
    # precision / perf
    bf16=use_bf16, fp16=use_fp16, tf32=use_tf32,
    dataloader_num_workers=8,
    dataloader_pin_memory=True,
    group_by_length=True,
    optim="adamw_torch_fused",
    # logging / eval / save
    logging_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=True,         # restore BLEU-best
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=3,
    save_safetensors=True,
    report_to=[],
    data_seed=42,
    **strategy_kwargs,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,          # future-proof vs tokenizer=
    data_collator=collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
eval_metrics = trainer.evaluate()
print("Eval:", eval_metrics)

### 7. Quick evaluation / generation sanity checks

In [ ]:
# 7) Inference helpers: beam + optional MBR re-ranking  (NL → EN)
import re
import unicodedata
import torch
from sacrebleu.metrics import CHRF

CTRL = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069]")

def _clean(s: str) -> str:
    s = CTRL.sub("", s or "")
    return unicodedata.normalize("NFKC", s).strip()

def _device():
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")

@torch.inference_mode()
def translate_nl2en(
    texts,
    num_beams: int = 5,
    max_new_tokens: int = 128,
    min_new_tokens: int = 4,
    no_repeat_ngram_size: int = 3,
    length_penalty: float = 1.1,
    repetition_penalty: float = 1.05,
    batch_size: int = 64,
):
    """
    High-quality decoding (NL → EN) with batching and safe defaults.
    - Uses beam search by default; if num_beams==1, switches to greedy and disables beam-only knobs.
    """
    model.eval()
    dev = _device()

    if isinstance(texts, str):
        texts = [texts]

    outs = []
    for i in range(0, len(texts), batch_size):
        chunk = [_clean(t) for t in texts[i : i + batch_size]]
        enc = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=256)
        enc = {k: v.to(dev) for k, v in enc.items()}

        # If beams==1, avoid beam-only params to keep config valid
        use_no_repeat = no_repeat_ngram_size if num_beams > 1 else 0
        lp = length_penalty if num_beams > 1 else 1.0

        # autocast for speed when on CUDA (bf16/fp16 handled by AMP)
        with torch.autocast(device_type="cuda", enabled=(dev.type == "cuda")):
            gen_ids = model.generate(
                **enc,
                num_beams=num_beams,
                do_sample=False,                 # deterministic for evaluation/production
                max_new_tokens=max_new_tokens,
                min_new_tokens=min_new_tokens,
                no_repeat_ngram_size=use_no_repeat,
                length_penalty=lp,
                repetition_penalty=repetition_penalty,
                use_cache=True,
            )
        outs.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))

    return [_clean(o) for o in outs]

# MBR decoding (reference-free): 1 strong beam + (K-1) samples; choose by mean chrF to others
@torch.inference_mode()
def translate_nl2en_mbr(
    texts,
    K: int = 10,               # total candidates per input (1 beam + K-1 samples)
    max_new_tokens: int = 128,
    top_p: float = 0.9,
    temperature: float = 0.7,
    batch_size: int = 32,
    seed: int | None = None,
    return_candidates: bool = False,    # if True, also return the n-best list per input
):
    """
    Reference-free MBR with chrF utility (NL → EN).
    - Mixes 1 strong beam with (K-1) diverse samples.
    - Picks the candidate with highest mean chrF vs. others.
    """
    model.eval()
    dev = _device()
    chrf = CHRF(word_order=2)

    if isinstance(texts, str):
        texts = [texts]
    cleaned = [_clean(t) for t in texts]
    results = []
    all_candidates = [] if return_candidates else None

    # Optional reproducible sampling
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    sK = max(1, K - 1)

    for start in range(0, len(cleaned), batch_size):
        batch = cleaned[start : start + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        enc = {k: v.to(dev) for k, v in enc.items()}

        with torch.autocast(device_type="cuda", enabled=(dev.type == "cuda")):
            # 1) One strong candidate via beam search
            beam_ids = model.generate(
                **enc,
                do_sample=False,
                num_beams=5,
                num_return_sequences=1,
                max_new_tokens=max_new_tokens,
                min_new_tokens=4,
                no_repeat_ngram_size=3,
                length_penalty=1.1,
                use_cache=True,
            )
            # 2) (K-1) sampled candidates (greedy base, sampled decoding)
            samp_ids = model.generate(
                **enc,
                do_sample=True,
                top_p=top_p,
                temperature=temperature,
                num_beams=1,
                num_return_sequences=sK,
                max_new_tokens=max_new_tokens,
                min_new_tokens=4,
                no_repeat_ngram_size=0,  # avoid overhead in sampling
                length_penalty=1.0,
                use_cache=True,
            )

        # Decode
        beam_txt = tokenizer.batch_decode(beam_ids, skip_special_tokens=True)
        samp_txt = tokenizer.batch_decode(samp_ids, skip_special_tokens=True)

        # Group per source: beam (1) + samples (sK)
        beam_groups = [beam_txt[i : i + 1] for i in range(len(beam_txt))]
        samp_groups = [samp_txt[i : i + sK] for i in range(0, len(samp_txt), sK)]

        # MBR select: candidate with highest mean chrF to others
        for beam1, samples in zip(beam_groups, samp_groups):
            cand_list = [_clean(t) for t in (beam1 + samples)]
            # de-duplicate while preserving order
            seen, uniq = set(), []
            for t in cand_list:
                if t and t not in seen:
                    seen.add(t)
                    uniq.append(t)
            if not uniq:
                results.append("")
                if return_candidates:
                    all_candidates.append([])
                continue

            scores = []
            for i, c in enumerate(uniq):
                others = uniq[:i] + uniq[i + 1 :]
                # mean chrF vs. others
                mean_u = float(
                    sum(chrf.sentence_score(c, [o]).score for o in others) / max(1, len(others))
                )
                scores.append(mean_u)

            best_idx = int(torch.tensor(scores).argmax().item())
            best = uniq[best_idx]
            results.append(best)
            if return_candidates:
                all_candidates.append(uniq)

    return (results, all_candidates) if return_candidates else results

### 8. BLEU / chrF evaluation (validate NL→EN)

In [ ]:
# 8) FAST BLEU/chrF evaluation (greedy/partial, much quicker) - NL→EN
import torch
from sacrebleu.metrics import BLEU, CHRF
from transformers import GenerationConfig
import random

# Align model & tokenizer token ids (safe)
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# Speed mode toggles
FAST = True                 # set False for full-strength eval
SAMPLE_N = 2000             # sample size if FAST=True

# Decoding knobs (FAST vs FULL)
if FAST:
    N = SAMPLE_N
    GEN_MAX_NEW = 96        # shorter decode
    NUM_BEAMS = 1           # greedy (fastest)
    BATCH_SIZE = 128        # bigger batch if VRAM allows
    NO_REPEAT = 0           # disable constraint for speed
else:
    N = None                # full validation set
    GEN_MAX_NEW = 128
    NUM_BEAMS = 5
    BATCH_SIZE = 64
    NO_REPEAT = 3

# Keep a consistent generation_config (we still pass args explicitly)
model.generation_config = GenerationConfig(
    max_new_tokens=GEN_MAX_NEW,
    num_beams=NUM_BEAMS,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

# Build (optionally sampled) eval slice
if N is None or N >= len(val_data):
    idxs = list(range(len(val_data)))
else:
    random.seed(42)
    idxs = random.sample(range(len(val_data)), k=N)

src_texts = [val_data[i]["translation"]["nl"] for i in idxs]   # NL source
references = [val_data[i]["translation"]["en"] for i in idxs]  # EN references

device = next(model.parameters()).device
hypotheses = []
model.eval()

# Optional: enable flash/mem-efficient attention where possible
try:
    torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=True)
except Exception:
    pass

@torch.inference_mode()
def _generate_batch(texts):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
    enc = {k: v.to(device) for k, v in enc.items()}

    # If beams==1, avoid beam-only knobs
    use_no_repeat = NO_REPEAT if NUM_BEAMS > 1 else 0
    length_penalty = 1.1 if NUM_BEAMS > 1 else 1.0

    # Let autocast pick the right dtype (fp16/bf16) when on CUDA
    with torch.autocast(device_type="cuda", enabled=(device.type == "cuda")):
        gen_ids = model.generate(
            **enc,
            num_beams=NUM_BEAMS,
            do_sample=False,                  # deterministic for eval
            max_new_tokens=GEN_MAX_NEW,
            min_new_tokens=2,
            no_repeat_ngram_size=use_no_repeat,
            length_penalty=length_penalty,
            early_stopping=False,
            use_cache=True,
        )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

for i in range(0, len(src_texts), BATCH_SIZE):
    batch_src = src_texts[i : i + BATCH_SIZE]
    outs = _generate_batch(batch_src)
    hypotheses.extend([s.strip() for s in outs])

bleu = BLEU(effective_order=True, smooth_method="exp", lowercase=True)
chrf = CHRF(word_order=2)

print(
    ("[FAST] " if FAST else "[FULL] ")
    + f"BLEU = {bleu.corpus_score(hypotheses, [references]).score:.2f} "
    + f"| chrF2 = {chrf.corpus_score(hypotheses, [references]).score:.2f} "
    + f"| n={len(src_texts)} | beams={NUM_BEAMS} | max_new={GEN_MAX_NEW} | bs={BATCH_SIZE}"
)

for s, hyp, ref in list(zip(src_texts, hypotheses, references))[:5]:
    print("\nNL:", s)
    print("EN(hyp):", hyp)
    print("EN(ref):", ref)

### 9. Save the fine-tuned NL→EN model + tokenizer

In [ ]:
# 9) Save model & tokenizer (NL→EN)
import os
save_dir = "../models/nl_en_l40s_final_v3"
os.makedirs(save_dir, exist_ok=True)

# Save weights, config, and tokenizer
model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

# Also persist the (sanitized) generation config used during eval/inference
try:
    gc.save_pretrained(save_dir)  # writes generation_config.json
except Exception:
    # Fallback for older versions
    try:
        gc.to_json_file(os.path.join(save_dir, "generation_config.json"))
    except Exception:
        pass

print("Saved to:", save_dir)

### 10. Batch translate a file (input NL → output EN)

In [ ]:
# 10) Translate a file (NL -> EN) — keep Sentence, NL, and add EN
from pathlib import Path
import pandas as pd
import torch

INPUT_PATH = Path("translations_en2nl_polished_final.csv")  # expects columns: Sentence, NL
OUTPUT_CSV = Path("translations_nl2en.csv")                 # semicolon-delimited
BATCH_SIZE = 64

# 1) Load
if INPUT_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(INPUT_PATH, dtype=str)
else:
    df = pd.read_csv(INPUT_PATH, sep=None, engine="python", dtype=str)

# 2) Validate required columns and prep text
required = {"Sentence", "NL"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Input is missing required column(s): {', '.join(sorted(missing))}")

df["Sentence"] = df["Sentence"].astype(str).fillna("")
df["NL"] = df["NL"].astype(str).fillna("")

sentences = df["NL"].tolist()

# 3) Translate in batches (beam). For MBR, swap the function below if desired.
def translate_batch_beam_nl2en(texts):
    model.eval()
    tok = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        ids = model.generate(
            **tok,
            num_beams=5,
            max_new_tokens=128,
            min_new_tokens=4,
            no_repeat_ngram_size=3,
            length_penalty=1.1,
            repetition_penalty=1.05,
        )
    return tokenizer.batch_decode(ids, skip_special_tokens=True)

outputs = []
for i in range(0, len(sentences), BATCH_SIZE):
    batch = sentences[i : i + BATCH_SIZE]
    outputs.extend(translate_batch_beam_nl2en(batch))
    # For extra quality (slower), use MBR instead:
    # outputs.extend(translate_nl2en_mbr(batch, K=10))

# 4) Save clean CSV (semicolon) with Sentence, NL, EN
def clean(s: str) -> str:
    return str(s).replace("\r", " ").replace("\n", " ").replace(";", ",").strip()

out = pd.DataFrame({
    "Sentence": [clean(s) for s in df["Sentence"].tolist()],
    "NL":       [clean(s) for s in df["NL"].tolist()],
    "EN":       [clean(t) for t in outputs],
})
out.to_csv(OUTPUT_CSV, sep=";", index=False, encoding="utf-8-sig")
print("Wrote:", OUTPUT_CSV)

### 11. Polishing

In [ ]:
# 11) Polishing (domain glossary + idioms + spacing) — NL terms → EN show style
import re
import unicodedata
import pandas as pd
from pathlib import Path

IN_CSV = "translations_nl2en.csv"
OUT_XLSX = "Final Translation.xlsx"   # <- Excel output

df = pd.read_csv(IN_CSV, sep=";", dtype=str)

# Preserve useful columns if present
for col in ["Sentence", "NL", "EN"]:
    if col not in df.columns:
        df[col] = ""

df["Sentence"] = df["Sentence"].astype(str)
df["NL"] = df["NL"].astype(str)
df["EN"] = df["EN"].astype(str)

def _norm(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = re.sub(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069]", "", text)  # strip invisibles
    return text.strip()

def sub_ci(text, pattern, repl):
    return re.sub(pattern, repl, text, flags=re.IGNORECASE)

def tidy_spaces(text):
    # punctuation spacing
    text = re.sub(r"\s+([,.:;!?…])", r"\1", text)
    text = re.sub(r"\s{2,}", " ", text)
    # normalize ellipsis / em-dash spacing
    text = re.sub(r"\s*…\s*", "…", text)
    text = re.sub(r"\s*—\s*", " — ", text)
    # remove space before apostrophes like " don’t" -> "don’t"
    text = re.sub(r"\s+'", "'", text)
    # collapse accidental dupes (simple pass)
    text = re.sub(r"\b(\w+)\s+\1\b", r"\1", text, flags=re.IGNORECASE)
    return text.strip()

# NL → EN Survivor glossary (longer/stricter patterns first)
GLOSS_NL2EN = [
    (r"\bonverwacht wegstemmen\b", "blindside"),
    (r"\bfinale stamraad\b", "final tribal council"),
    (r"\bverborgen immuniteitsbeeldje\b", "hidden immunity idol"),
    (r"\bimmuniteitsbeeldje\b", "immunity idol"),
    (r"\bimmuniteitsproef\b", "immunity challenge"),
    (r"\bbeloningsproef\b", "reward challenge"),
    (r"\bsamensmelting\b", "merge"),
    (r"\bwegstemmen\b", "vote out"),
    (r"\balliantie\b", "alliance"),
    (r"\bstamraad\b", "tribal council"),
    (r"\bstam(men)?\b", "tribe"),
    (r"\bdeelnemer(s)?\b", "castaway"),
    (r"\bimmuniteit\b", "immunity"),
    (r"\bkamp\b", "camp"),
    (r"\bfakkel\b", "torch"),
    (r"\bproef\b", "challenge"),
    (r"^survivors,?\s*klaar\??$", "Survivors, ready?"),
    (r"^start!?$", "Go!"),
]

# Light style fixes for English outputs
STYLE_EN = [
    (r"\bok[eé]\b", "OK"),
    (r"\.\.\.", "…"),
    (r"\bi am not even\b", "I'm not even"),
    (r"\bi do not\b", "I don't"),
    (r"\bwe are\b", "we're"),
    (r"\bis not\b", "isn't"),
    (r"\bare not\b", "aren't"),
]

def polish_en(text: str) -> str:
    if not text:
        return text
    text = _norm(text)
    for pat, rep in GLOSS_NL2EN:
        text = sub_ci(text, pat, rep)
    for pat, rep in STYLE_EN:
        text = sub_ci(text, pat, rep)
    return tidy_spaces(text)

df["EN"] = df["EN"].map(polish_en)

# Keep Sentence + NL + EN for side-by-side QA
df_out = df[["Sentence", "NL", "EN"]]

# Write a clean Excel workbook (each column remains separate)
# Requires openpyxl; if missing: pip install openpyxl
df_out.to_excel(OUT_XLSX, index=False, sheet_name="Translations")

print(f"Wrote: {OUT_XLSX}")